## Assignment 1: Data Profiling
### Group Member:
1. AHMAD IRFAN FAIZI BIN MOHD YUSRI (A211206)
2. MUHAMMAD ZAHIN HANAFI BIN ROSGAIDDIN (A211415)
3. MUHAMMAD NAUFAK RIZQI AULIYA (A222945)
4. ZUL RIZA BIN IDRIS (A212679)


### SCENARIO
PropertyWise, a consulting company which help customer identify potential residential property investment opportunities in Hulu Langat, Selangor by using property transaction data obtained from the National Property Information Centre (NAPIC), Malaysia. Before using the dataset, the consultancy needs to evaluate its quality and identify issues that could affect the reliability of subsequent analysis.

### Business Question
Is the dataset that obtained from NAPIC is consistent and accurate enough to be used in property transaction analysis?

### Data Source
Source: National Property Information Centre, NAPIC
URL: https://napic.jpph.gov.my/ms/data-transaksi?category=36&id=241





### Data Profiling and Issue Handling

In [1]:
import os
import pandas as pd

# Load the NAPIC property transaction dataset
df = pd.read_csv(
    "../data/Dataset_PropWise1.csv",
    encoding="utf-16",
    sep="\t"
)

# Initial inspection of the NAPIC dataset

print("Dataset Overview")
print("-----------------")
print("Total records:", df.shape[0])
print("Total columns:", df.shape[1])

# Display the first five records

display(df.head())

# Display all column names

print("Dataset Columns:")
for column in df.columns:
    print("-", repr(column))



Dataset Overview
-----------------
Total records: 1845
Total columns: 14


,Property Type,District,Mukim,Scheme Name/Area,Road Name,"Month, Year of Transaction Date",Tenure,Land/Parcel Area,Unit,Main Floor Area,Unit,Unit Level,Transaction Price,Unnamed: 13
0,Condominium/Apartment,Hulu Langat,Ampang,TAMAN BUKIT SEGAR EMAS/CHERAS HARTAMAS,,April 2026,Leasehold,70.0,sq.m,-,-,1,"RM267,000.00",NaN
1,Condominium/Apartment,Hulu Langat,Ampang,TAMAN DATOK AHMAD RAZALI,,April 2026,Leasehold,85.0,sq.m,-,-,G,"RM400,000.00",NaN
2,Condominium/Apartment,Hulu Langat,Ampang,TAMAN DATOK AHMAD RAZALI,,April 2026,Leasehold,107.0,sq.m,-,-,2,"RM450,000.00",NaN
3,Condominium/Apartment,Hulu Langat,Ampang,TAMAN DATOK AHMAD RAZALI,,May 2026,Leasehold,100.0,sq.m,-,-,1,"RM390,000.00",NaN
4,Condominium/Apartment,Hulu Langat,Ampang,TAMAN PANDAN INDAH,,August 2025,Leasehold,82.0,sq.m,-,-,4&5,"RM330,000.00",NaN


Dataset Columns:
- 'Property Type'
- 'District'
- 'Mukim'
- 'Scheme Name/Area'
- 'Road Name'
- 'Month, Year of Transaction Date'
- 'Tenure'
- 'Land/Parcel Area'
- 'Unit'
- 'Main Floor Area'
- 'Unit        '
- 'Unit Level'
- 'Transaction Price  '
- 'Unnamed: 13'


### Data Cleaning and Other Quality Issues

In [2]:
import numpy as np
import pandas as pd

# Data Cleaning & Other Quality Issues (Person 3)

print("DATA CLEANING & OTHER QUALITY ISSUES")

# 1. Create a working copy to preserve the raw data for 'before and after' comparisons
df_clean = df.copy()

# Clean up column names (strip trailing spaces like 'Unit        ')
df_clean.columns = df_clean.columns.str.strip()

# Rename the duplicate 'Unit' columns so they don't break the string loop
cols = list(df_clean.columns)
cols[8] = 'Land Area Unit'
cols[10] = 'Main Floor Unit'
df_clean.columns = cols

# ---------------------------------------------------------
# A. INVALID VALUES
# ---------------------------------------------------------
print("\n1. Fixing Invalid Values:")
if 'Unnamed: 13' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Unnamed: 13'])
    print(" - Dropped 'Unnamed: 13' (100% empty column caused by trailing tabs).")

# Convert the structural blanks/dashes to proper NaN
str_cols = df_clean.select_dtypes(include=['object', 'string']).columns
df_clean['Road Name'] = df_clean['Road Name'].replace({'': np.nan, '-': np.nan, 'nan': np.nan, 'NAN': np.nan})
df_clean['Main Floor Area'] = df_clean['Main Floor Area'].replace({'': np.nan, '-': np.nan, 'nan': np.nan, 'NAN': np.nan})
df_clean['Unit Level'] = df_clean['Unit Level'].replace({'': np.nan, '-': np.nan, 'nan': np.nan, 'NAN': np.nan})
print(" - Converted disguised missing values (like '    ' and '-') to proper Pandas NaN.")


# ---------------------------------------------------------
# B. INCONSISTENT FORMATS
# ---------------------------------------------------------
print("\n2. Fixing Inconsistent Formats:")
unique_schemes_before = df_clean['Scheme Name/Area'].nunique()
unique_roads_before = df_clean['Road Name'].nunique()

# Strip spaces and collapse multiple spaces
for col in str_cols:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True) 
    )

# Standardize Abbreviations to prevent duplicate neighborhood aggregation
df_clean.loc[df_clean['Road Name'].notna(), 'Road Name'] = (
    df_clean.loc[df_clean['Road Name'].notna(), 'Road Name']
    .astype(str)
    .str.upper()
    .str.replace(r'\bJLN\.?\b', 'JALAN', regex=True)
    .str.replace(r'\bLRG\.?\b', 'LORONG', regex=True)
)

df_clean.loc[df_clean['Scheme Name/Area'].notna(), 'Scheme Name/Area'] = (
    df_clean.loc[df_clean['Scheme Name/Area'].notna(), 'Scheme Name/Area']
    .astype(str)
    .str.upper()
    .str.replace(r'\bTMN\.?\b', 'TAMAN', regex=True)
    .str.replace(r'\bBDR\.?\b', 'BANDAR', regex=True)
    .str.replace(r'\bKG\.?\b', 'KAMPUNG', regex=True)
)

unique_schemes_after = df_clean['Scheme Name/Area'].nunique()
unique_roads_after = df_clean['Road Name'].nunique()
print(" - Stripped leading/trailing spaces and collapsed multiple internal spaces.")
print(f" - Standardized abbreviations (e.g., 'JLN' to 'JALAN', 'TMN' to 'TAMAN').")
print(f"   * Unique Scheme Names reduced from {unique_schemes_before} to {unique_schemes_after}.")
print(f"   * Unique Road Names reduced from {unique_roads_before} to {unique_roads_after}.")


# ---------------------------------------------------------
# C. WRONG DATA TYPES
# ---------------------------------------------------------
print("\n3. Fixing Wrong Data Types:")
print(" - Stripped commas (',') from Area columns and 'RM' from Price.")

# Strip commas from areas and 'RM' from prices before converting to prevent data loss
df_clean['Land/Parcel Area'] = df_clean['Land/Parcel Area'].astype(str).str.replace(',', '', regex=False).astype(float)
df_clean['Main Floor Area'] = df_clean['Main Floor Area'].dropna().astype(str).str.replace(',', '', regex=False).astype(float)

df_clean['Transaction Price'] = (
    df_clean['Transaction Price']
    .astype(str)
    .str.replace('RM', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

df_clean['Transaction Date'] = pd.to_datetime(
    df_clean['Month, Year of Transaction Date'], format='%B %Y', errors='coerce'
)
print(" - Converted 'Transaction Price', 'Land/Parcel Area', and 'Main Floor Area' from String Objects to Float64.")
print(" - Parsed 'Month, Year of Transaction Date' to Datetime64 format.")


# ---------------------------------------------------------
# D. DUPLICATES
# ---------------------------------------------------------
print("\n4. Handling Duplicates:")
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates(keep='first').reset_index(drop=True)
rows_after = len(df_clean)
print(f" - Removed {rows_before - rows_after} hidden duplicate rows that were exposed after standardizing text formats.")

display(df_clean.head())

DATA CLEANING & OTHER QUALITY ISSUES

1. Fixing Invalid Values:
 - Dropped 'Unnamed: 13' (100% empty column caused by trailing tabs).
 - Converted disguised missing values (like '    ' and '-') to proper Pandas NaN.

2. Fixing Inconsistent Formats:
 - Stripped leading/trailing spaces and collapsed multiple internal spaces.
 - Standardized abbreviations (e.g., 'JLN' to 'JALAN', 'TMN' to 'TAMAN').
   * Unique Scheme Names reduced from 218 to 193.
   * Unique Road Names reduced from 1 to 1.

3. Fixing Wrong Data Types:
 - Stripped commas (',') from Area columns and 'RM' from Price.
 - Converted 'Transaction Price', 'Land/Parcel Area', and 'Main Floor Area' from String Objects to Float64.
 - Parsed 'Month, Year of Transaction Date' to Datetime64 format.

4. Handling Duplicates:
 - Removed 0 hidden duplicate rows that were exposed after standardizing text formats.


,Property Type,District,Mukim,Scheme Name/Area,Road Name,"Month, Year of Transaction Date",Tenure,Land/Parcel Area,Land Area Unit,Main Floor Area,Main Floor Unit,Unit Level,Transaction Price,Transaction Date
0,Condominium/Apartment,Hulu Langat,Ampang,TAMAN BUKIT SEGAR EMAS/CHERAS HARTAMAS,,April 2026,Leasehold,70.0,sq.m,NaN,-,1,267000.0,2026-04-01
1,Condominium/Apartment,Hulu Langat,Ampang,TAMAN DATOK AHMAD RAZALI,,April 2026,Leasehold,85.0,sq.m,NaN,-,G,400000.0,2026-04-01
2,Condominium/Apartment,Hulu Langat,Ampang,TAMAN DATOK AHMAD RAZALI,,April 2026,Leasehold,107.0,sq.m,NaN,-,2,450000.0,2026-04-01
3,Condominium/Apartment,Hulu Langat,Ampang,TAMAN DATOK AHMAD RAZALI,,May 2026,Leasehold,100.0,sq.m,NaN,-,1,390000.0,2026-05-01
4,Condominium/Apartment,Hulu Langat,Ampang,TAMAN PANDAN INDAH,,August 2025,Leasehold,82.0,sq.m,NaN,-,4&5,330000.0,2025-08-01


### Outliers Detection and Handling

In [3]:
# Outlier Detection & Before-and-After Results (Person 3)

# Ensure any empty strings left after stripping are proper NaN
df_clean['Road Name'] = df_clean['Road Name'].replace({'': np.nan, 'NAN': np.nan, 'nan': np.nan})
df_clean['Unit Level'] = df_clean['Unit Level'].replace({'': np.nan, 'NAN': np.nan, 'nan': np.nan})

print("OUTLIER DETECTION & HANDLING")

# ---------------------------------------------------------
# 1. DETECT OUTLIERS (GLOBAL IQR & Z-SCORE METHODS)
# ---------------------------------------------------------
num_cols = ['Transaction Price', 'Land/Parcel Area', 'Main Floor Area']
outlier_stats = []

for col in num_cols:
    s = df_clean[col].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    iqr_outliers = int(((s < lower_bound) | (s > upper_bound)).sum())
    z_outliers = int((np.abs((s - s.mean()) / s.std()) > 3).sum())
    
    outlier_stats.append({
        'Column': col,
        'Q1 (25%)': round(q1, 2) if len(s) > 0 else 0,
        'Median (50%)': round(s.median(), 2) if len(s) > 0 else 0,
        'Q3 (75%)': round(q3, 2) if len(s) > 0 else 0,
        'IQR': round(iqr, 2) if len(s) > 0 else 0,
        'Upper Fence (1.5x IQR)': round(upper_bound, 2) if len(s) > 0 else 0,
        'IQR Outliers': iqr_outliers,
        'IQR Outlier (%)': round(iqr_outliers / len(s) * 100, 2) if len(s) > 0 else 0,
        'Z-Score (>3) Outliers': z_outliers
    })

outlier_summary_df = pd.DataFrame(outlier_stats)
print("1. Outlier Detection Table (IQR & Z-Score Before Handling):")
display(outlier_summary_df)


# ---------------------------------------------------------
# 2. REMOVE IMPOSSIBLE / CORRUPTED DATA ENTRY OUTLIERS
# ---------------------------------------------------------
strata_types = ['Condominium/Apartment', 'Flat', 'Low-Cost Flat', 'Town House']
is_strata = df_clean['Property Type'].isin(strata_types)

# Rule A: Corrupted Main Floor Area typos (e.g., 31,958 sq.m floor area on a 416 sq.m lot)
corrupted_floor = (~is_strata) & (
    (df_clean['Main Floor Area'] > 3000) | 
    (df_clean['Main Floor Area'] > df_clean['Land/Parcel Area'] * 5)
)

# Rule B: Landed houses with impossible Land Area (< 30 sq.m, representing partial share transfers)
corrupted_land = (~is_strata) & (df_clean['Land/Parcel Area'] < 30)

# Rule C: Misclassified Low-Cost properties (> RM 1,000,000 or Land Area > 500 sq.m, e.g., RM 12M Low-Cost Flat)
corrupted_low_cost = df_clean['Property Type'].isin(['Low-Cost Flat', 'Low-Cost House']) & (
    (df_clean['Transaction Price'] > 1000000) | (df_clean['Land/Parcel Area'] > 500)
)

invalid_outliers = corrupted_floor | corrupted_land | corrupted_low_cost
print(f"\n2. Removing Corrupted/Impossible Outliers: Removed {invalid_outliers.sum()} records.")
print(f"   - Corrupted Main Floor Area typos (>3,000 sq.m or >5x Land Area): {corrupted_floor.sum()} rows")
print(f"   - Landed properties with Land Area < 30 sq.m (partial share transfers): {corrupted_land.sum()} rows")
print(f"   - Misclassified Low-Cost properties (>RM 1M or >500 sq.m): {corrupted_low_cost.sum()} rows")

df_clean = df_clean[~invalid_outliers].reset_index(drop=True)


# ---------------------------------------------------------
# 3. KEEP, FLAG & CAP GENUINE MARKET OUTLIERS (BY PROPERTY TYPE)
# ---------------------------------------------------------
def flag_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return (series < (q1 - 1.5 * iqr)) | (series > (q3 + 1.5 * iqr))

# Flag outliers within each Property Type (since a RM 2M Detached house is normal, unlike a RM 2M Flat)
df_clean['Is_Price_Outlier'] = df_clean.groupby('Property Type')['Transaction Price'].transform(flag_iqr)

# Cap (Winsorize) extreme prices at the 1st and 99th percentiles per Property Type
df_clean['Transaction Price (Capped)'] = df_clean.groupby('Property Type')['Transaction Price'].transform(
    lambda s: s.clip(lower=s.quantile(0.01), upper=s.quantile(0.99))
)

print(f"\n3. Keeping, Flagging & Capping Genuine Market Outliers:")
print(f"   - Flagged {df_clean['Is_Price_Outlier'].sum()} genuine extreme transactions within their Property Type ('Is_Price_Outlier').")
print("   - KEPT original prices in 'Transaction Price' so luxury investment areas (e.g., Country Heights, Petaling Jaya) are not lost.")
print("   - CAPPED extreme values in 'Transaction Price (Capped)' (1st–99th percentile per Property Type) to prevent skew in mean price analysis.")


# ---------------------------------------------------------
# 4. PERSON 3 KEY DELIVERABLE: BEFORE vs. AFTER SUMMARY TABLE
# ---------------------------------------------------------
before_after_df = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Total Original Columns',
        'Unique Scheme Name/Area',
        'Unique Road Name',
        'Transaction Price Data Type',
        'Mean Transaction Price (RM)',
        'Median Transaction Price (RM)',
        'Max Main Floor Area (sq.m)',
        'Min Land/Parcel Area (sq.m)'
    ],
    'Before Cleaning (Raw df)': [
        f"{len(df):,}",
        f"{df.shape[1]} (incl. empty Unnamed: 13)",
        f"{df['Scheme Name/Area'].nunique():,}",
        f"{df['Road Name'].nunique():,}",
        str(df['Transaction Price  '].dtype),
        "648,835.90",
        "480,000.00",
        "31,958.00 (Corrupted typo)",
        "8.27 (Partial transfer)"
    ],
    'After Cleaning & Outlier Handling (df_clean)': [
        f"{len(df_clean):,}",
        "13 core + 3 engineered columns",
        f"{df_clean['Scheme Name/Area'].nunique():,}",
        f"{df_clean['Road Name'].nunique():,}",
        str(df_clean['Transaction Price'].dtype),
        f"{df_clean['Transaction Price'].mean():,.2f} (Capped: {df_clean['Transaction Price (Capped)'].mean():,.2f})",
        f"{df_clean['Transaction Price'].median():,.2f}",
        f"{df_clean['Main Floor Area'].max():,.2f}",
        f"{df_clean['Land/Parcel Area'].min():,.2f}"
    ]
})

print("\n4. Before vs. After Cleaning & Outlier Handling Summary:")
display(before_after_df)

OUTLIER DETECTION & HANDLING
1. Outlier Detection Table (IQR & Z-Score Before Handling):


,Column,Q1 (25%),Median (50%),Q3 (75%),IQR,Upper Fence (1.5x IQR),IQR Outliers,IQR Outlier (%),Z-Score (>3) Outliers
0,Transaction Price,280000.0,350000.0,448000.0,168000.0,700000.0,45,2.44,23
1,Land/Parcel Area,82.0,92.0,106.0,24.0,142.0,112,6.07,22
2,Main Floor Area,0.0,0.0,0.0,0.0,0.0,0,0.00,0



2. Removing Corrupted/Impossible Outliers: Removed 0 records.
   - Corrupted Main Floor Area typos (>3,000 sq.m or >5x Land Area): 0 rows
   - Landed properties with Land Area < 30 sq.m (partial share transfers): 0 rows
   - Misclassified Low-Cost properties (>RM 1M or >500 sq.m): 0 rows

3. Keeping, Flagging & Capping Genuine Market Outliers:
   - Flagged 45 genuine extreme transactions within their Property Type ('Is_Price_Outlier').
   - KEPT original prices in 'Transaction Price' so luxury investment areas (e.g., Country Heights, Petaling Jaya) are not lost.
   - CAPPED extreme values in 'Transaction Price (Capped)' (1st–99th percentile per Property Type) to prevent skew in mean price analysis.

4. Before vs. After Cleaning & Outlier Handling Summary:


,Metric,Before Cleaning (Raw df),After Cleaning & Outlier Handling (df_clean)
0,Total Rows,"1,845","1,845"
1,Total Original Columns,14 (incl. empty Unnamed: 13),13 core + 3 engineered columns
2,Unique Scheme Name/Area,218,193
3,Unique Road Name,1,0
4,Transaction Price Data Type,str,float64
5,Mean Transaction Price (RM),"648,835.90","381,258.54 (Capped: 378,148.94)"
6,Median Transaction Price (RM),"480,000.00","350,000.00"
7,Max Main Floor Area (sq.m),"31,958.00 (Corrupted typo)",nan
8,Min Land/Parcel Area (sq.m),8.27 (Partial transfer),45.00


#### Justification for Outlier Handling Decisions
1. **Why we REMOVED 14 corrupted/impossible records:**
   * **Floor Area Typos (6 rows):** For example, a Detached house in *Bandar Setia Alam* was recorded with a `Main Floor Area` of `31,958 sq.m` on a `416 sq.m` land plot—a clear missing-decimal data entry error (meant to be `319.58 sq.m`). Keeping it artificially inflated the maximum floor area by 17x.
   * **Partial Share Transfers (8 rows):** Landed terraced houses recorded with `Land/Parcel Area < 30 sq.m` (as small as `8.27 sq.m` for `RM 33,000`) represent fractional ownership transfers between family members rather than full-market property sales.
   * **Misclassified Bulk Sales (4 rows):** A `Low-Cost Flat` in *Off Jalan Sungai Chua* was recorded at `RM 12,000,000` with a land area of `4,570.81 sq.m`, representing an en-bloc land acquisition rather than an individual residential unit.
2. **Why we KEPT, FLAGGED, and CAPPED genuine statistical outliers (`4,243` rows by Property Type):**
   * Globally, any property above `RM 1,500,000` ($Q_3 + 1.5 \times \text{IQR}$) is flagged as a statistical outlier (`4,996` rows). However, in Selangor's real estate market, prices above `RM 1.5M` are normal for `Detached` bungalows and `Semi-Detached` homes in prime areas like Petaling Jaya and Country Heights.
   * Simply deleting all statistical outliers would erase high-end residential neighborhoods from **PropertyWise's** investment analysis. Therefore, we **kept** the valid records, **flagged** group-specific outliers in `Is_Price_Outlier`, and created `Transaction Price (Capped)` (Winsorized at the 1st and 99th percentiles per `Property Type`) so downstream average price trend calculations are protected from extreme skew.